# Interlocked Credit in Bangladesh's Rice Trading Sector — Analysis

This notebook produces every data-driven table and figure in Sections 3–6 and the appendix of
`paper/main.tex`. Outputs go to `paper/tables/` and `paper/figures/`.
Figure 4 (district map), Figure 5 and Table 6 (credit determinants) come from the scripts in this folder.

**Data.** The analysis uses the cleaned trader modules of the 2018 Bangladesh Millers and Traders
Survey (MATS). They are not distributed with this repository — see `data/README.md`.
Point the `MATS_DATA_DIR` environment variable at the folder that contains `trader_data/clean/`.

**Run.** From the repository root: `make analysis` (or *Run All* here). The notebook is committed
without outputs; please clear outputs before committing changes (`make check` enforces this).

In [ ]:
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

from scipy.stats import chi2_contingency

from common import (FIGURES, MIN_TRADERS, PRICE_MAX, PRICE_MIN, TABLES, fmt_c,
                    fmt_se, pct_of_answered, trader_file, write_tex)

warnings.filterwarnings('ignore')

# Interlocked Credit and Commodity Transactions

**Research question:** How prevalent are advance payment arrangements between traders and their paddy suppliers, and do they depress procurement prices paid to farmers?

**Step 1 — Descriptive statistics:** Credit arrangement types by trader type, upazila, and season.

Variables used (from `traders_modD_D9-D10.dta`):
| Variable | Label |
|----------|-------|
| `d35` | Did any suppliers let you buy on credit? (paddy) — None / Some / All |
| `d36` | Credit duration in days (paddy) |
| `d37` | Price premium for credit purchases (paddy) — Yes / No |
| `d38` | Price difference (BDT/kg, paddy) |
| `d39` | % purchases — payment on delivery (paddy) |
| `d40` | % purchases — advance payment *to traders* (paddy) |
| `d41` | % purchases — advance payment *to farmers* (paddy) |
| `d42–d47` | Same variables for rice |

In [ ]:
# ── Load trader credit module (D9-D10) and apply value labels ──────────────
credit_path = trader_file('traders_modD_D9-D10.dta')

df_cr = pd.read_stata(credit_path, convert_categoricals=False)

# Value label maps (from StataReader)
season_map  = {1: 'Aman 2016/17', 2: 'Boro 2017', 3: 'Aus 2017'}
b09_map     = {1: 'Paddy', 2: 'Rice', 3: 'Both'}
b10_map     = {1: 'Faria/bepari', 2: 'Aratdar', 3: 'Broker/intermediary', 4: 'Other'}
b11_map     = {1: 'Aratdar', 2: 'Broker/intermediary', 3: 'Wholesaler', 4: 'Retailer', 5: 'Other'}
credit_map  = {1: 'None of suppliers', 2: 'Some suppliers', 3: 'All suppliers'}
yn_map      = {1: 'Yes', 2: 'No'}
district_map = {
    4:'Barguna', 6:'Barisal', 9:'Bhola', 42:'Jhalokati', 78:'Patuakhali', 79:'Pirojpur',
    3:'Bandarban', 12:'Brahmanbaria', 13:'Chandpur', 15:'Chittagong', 19:'Comilla',
    22:"Cox's Bazar", 30:'Feni', 46:'Khagrachhari', 51:'Lakshmipur', 75:'Noakhali',
    84:'Rangamati', 26:'Dhaka', 29:'Faridpur', 33:'Gazipur', 35:'Gopalganj',
    48:'Kishoregonj', 54:'Madaripur', 56:'Manikganj', 59:'Munshiganj', 67:'Narayanganj',
    68:'Narsingdi', 82:'Rajbari', 86:'Shariatpur', 93:'Tangail', 1:'Bagerhat',
    18:'Chuadanga', 41:'Jessore', 44:'Jhenaidah', 47:'Khulna', 50:'Kushtia',
    55:'Magura', 57:'Meherpur', 65:'Narail', 87:'Satkhira', 39:'Jamalpur',
    61:'Mymensingh', 72:'Netrakona', 89:'Sherpur', 10:'Bogra', 38:'Joypurhat',
    64:'Naogaon', 69:'Natore', 70:'Chapai Nababganj', 76:'Pabna', 81:'Rajshahi',
    88:'Sirajganj', 27:'Dinajpur', 32:'Gaibandha', 49:'Kurigram', 52:'Lalmonirhat',
    73:'Nilphamari', 77:'Panchagarh', 85:'Rangpur', 94:'Thakurgaon', 36:'Habiganj',
    58:'Maulvibazar', 90:'Sunamganj', 91:'Sylhet'
}

# Upazila name from Stata value labels
with pd.io.stata.StataReader(credit_path) as _reader:
    upazila_map = _reader.value_labels().get('a05', {})

df_cr['season']       = df_cr['season'].map(season_map)
df_cr['commodity']    = df_cr['b09'].map(b09_map)
df_cr['trader_type']  = df_cr['b10'].map(b10_map)
df_cr['trader_role']  = df_cr['b11'].map(b11_map)
df_cr['district']     = df_cr['a04'].map(district_map)
df_cr['upazila']      = df_cr['a05'].map(upazila_map)

# Binary: any credit from suppliers
df_cr['any_credit_paddy'] = df_cr['d35'].map({1: 0, 2: 1, 3: 1})
df_cr['any_credit_rice']  = df_cr['d42'].map({1: 0, 2: 1, 3: 1})

# Credit scope labels
df_cr['credit_scope_paddy'] = df_cr['d35'].map(credit_map)
df_cr['credit_scope_rice']  = df_cr['d42'].map(credit_map)

# Price premium indicators
df_cr['price_premium_paddy'] = df_cr['d37'].map({1: 1, 2: 0})
df_cr['price_premium_rice']  = df_cr['d44'].map({1: 1, 2: 0})

print(f"Trader-season observations : {len(df_cr):,}")
print(f"Unique traders             : {df_cr['tempid'].nunique():,}")
print(f"\nSeason counts:\n{df_cr['season'].value_counts()}")
print(f"\nTrader type (b10):\n{df_cr['trader_type'].value_counts()}")
print(f"\nCommodity traded (b09):\n{df_cr['commodity'].value_counts()}")

## Table 1 — Credit arrangement types by trader type

In [ ]:
# ── Table 1A: Credit scope (None / Some / All suppliers) by trader type ─────
# Paddy rows only (d35); restrict to traders who deal in paddy
paddy_mask = df_cr['b09'].isin([1, 3])  # Paddy or Both
rice_mask  = df_cr['b09'].isin([2, 3])  # Rice or Both

def credit_scope_table(df, scope_col, group_col, label):
    ct = (
        df.dropna(subset=[scope_col, group_col])
          .groupby([group_col, scope_col])
          .size()
          .unstack(scope_col)
          .fillna(0)
          .astype(int)
    )
    # reorder columns if present
    col_order = [c for c in ['None of suppliers', 'Some suppliers', 'All suppliers'] if c in ct.columns]
    ct = ct[col_order]
    ct['Total'] = ct.sum(axis=1)
    # pct columns
    for col in col_order:
        ct[f'{col} (%)'] = (ct[col] / ct['Total'] * 100).round(1)
    print(f"\n{label}")
    print(ct.to_string())
    return ct

tbl1a = credit_scope_table(
    df_cr[paddy_mask], 'credit_scope_paddy', 'trader_type',
    'Credit from paddy suppliers by trader type (D35)'
)

tbl1b = credit_scope_table(
    df_cr[rice_mask], 'credit_scope_rice', 'trader_type',
    'Credit from rice suppliers by trader type (D42)'
)

In [ ]:
# ── Table 1B: Payment arrangement shares + credit terms by trader type ───────
# d39 = % payment on delivery, d40 = % advance to traders, d41 = % advance to farmers
# d36 = credit duration (days), d37 binary = paid price premium

def payment_stats_by_group(df, group_col, label):
    cols = {
        'd39': 'Payment on delivery (%)',
        'd40': 'Advance to traders (%)',
        'd41': 'Advance to farmers (%)',
        'd36': 'Credit duration (days)',
    }
    agg = {}
    for var, name in cols.items():
        sub = df.dropna(subset=[var, group_col])
        agg[name] = sub.groupby(group_col)[var].agg(['mean', 'median', 'count'])
        agg[name].columns = [f'{name} — mean', f'{name} — median', 'N']

    # Price premium: % saying Yes
    pp = df.dropna(subset=['price_premium_paddy', group_col])
    pp_rate = pp.groupby(group_col)['price_premium_paddy'].agg(
        **{'Price premium (%) — mean': 'mean', 'N_pp': 'count'}
    )
    pp_rate['Price premium (%) — mean'] = (pp_rate['Price premium (%) — mean'] * 100).round(1)

    result = pd.concat([v.drop(columns='N') for v in agg.values()], axis=1)
    result['Price premium (%) — mean'] = pp_rate['Price premium (%) — mean']

    # Round means and medians
    mean_cols = [c for c in result.columns if '— mean' in c]
    med_cols  = [c for c in result.columns if '— median' in c]
    result[mean_cols] = result[mean_cols].round(1)
    result[med_cols]  = result[med_cols].round(1)

    print(f"\n{label}")
    print(result.to_string())
    return result

tbl1c = payment_stats_by_group(
    df_cr[paddy_mask], 'trader_type',
    'Payment arrangement shares & credit terms — paddy, by trader type'
)

## Table 2 — Credit arrangement types by season

In [ ]:
# ── Table 2A: Credit scope by season ────────────────────────────────────────
tbl2a = credit_scope_table(
    df_cr[paddy_mask], 'credit_scope_paddy', 'season',
    'Credit from paddy suppliers by season (D35)'
)

tbl2b = credit_scope_table(
    df_cr[rice_mask], 'credit_scope_rice', 'season',
    'Credit from rice suppliers by season (D42)'
)

# ── Table 2B: Payment shares + credit terms by season ───────────────────────
tbl2c = payment_stats_by_group(
    df_cr[paddy_mask], 'season',
    'Payment arrangement shares & credit terms — paddy, by season'
)

## Table 3 — Credit arrangement types by district (upazila aggregated)

Upazila-level breakdowns produce ~200+ rows, so results are aggregated to district for readability. The upazila-level table is produced at the end for export.

In [ ]:
# ── Table 3A: Credit scope by district ──────────────────────────────────────
tbl3a = credit_scope_table(
    df_cr[paddy_mask], 'credit_scope_paddy', 'district',
    'Credit from paddy suppliers by district (D35)'
)

# ── Table 3B: Payment shares by district ────────────────────────────────────
tbl3b = payment_stats_by_group(
    df_cr[paddy_mask], 'district',
    'Payment arrangement shares & credit terms — paddy, by district'
)

# ── Table 3C: Upazila-level credit prevalence (% with any credit, paddy) ────
upazila_credit = (
    df_cr[paddy_mask]
    .dropna(subset=['any_credit_paddy', 'upazila'])
    .groupby('upazila')
    .agg(
        N=('any_credit_paddy', 'count'),
        n_traders=('tempid', 'nunique'),
        pct_any_credit=('any_credit_paddy', lambda x: round(x.mean() * 100, 1)),
        mean_credit_days=('d36', 'mean'),
        mean_pct_delivery=('d39', 'mean'),
        mean_pct_adv_traders=('d40', 'mean'),
        mean_pct_adv_farmers=('d41', 'mean'),
    )
    .round(1)
    .sort_index()
    .sort_values('pct_any_credit', ascending=False, kind='stable')
)

print("\nUpazila-level credit prevalence (paddy, sorted by % with any supplier credit):")
print(upazila_credit.to_string())

## Summary charts — Credit prevalence by trader type and season

In [ ]:
# ── Charts: credit prevalence and payment mix ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Interlocked Credit — Descriptive Statistics (Paddy, Traders Survey 2018)',
             fontsize=13, fontweight='bold')

# ── Panel A: % with any supplier credit, by trader type ─────────────────────
pct_by_type = (
    df_cr[paddy_mask]
    .dropna(subset=['any_credit_paddy', 'trader_type'])
    .groupby('trader_type')['any_credit_paddy']
    .agg(['mean', 'count'])
    .reset_index()
)
pct_by_type['mean'] *= 100
axes[0, 0].barh(pct_by_type['trader_type'], pct_by_type['mean'], color='steelblue')
axes[0, 0].set_xlabel('% of trader-seasons with any supplier credit')
axes[0, 0].set_title('A. Credit prevalence by trader type (paddy)')
for i, (v, n) in enumerate(zip(pct_by_type['mean'], pct_by_type['count'])):
    axes[0, 0].text(v + 0.5, i, f'{v:.1f}%  (N={n})', va='center', fontsize=9)
axes[0, 0].set_xlim(0, 105)

# ── Panel B: % with any supplier credit, by season ──────────────────────────
pct_by_season = (
    df_cr[paddy_mask]
    .dropna(subset=['any_credit_paddy', 'season'])
    .groupby('season')['any_credit_paddy']
    .agg(['mean', 'count'])
    .reset_index()
)
pct_by_season['mean'] *= 100
axes[0, 1].barh(pct_by_season['season'], pct_by_season['mean'], color='coral')
axes[0, 1].set_xlabel('% of trader-seasons with any supplier credit')
axes[0, 1].set_title('B. Credit prevalence by season (paddy)')
for i, (v, n) in enumerate(zip(pct_by_season['mean'], pct_by_season['count'])):
    axes[0, 1].text(v + 0.5, i, f'{v:.1f}%  (N={n})', va='center', fontsize=9)
axes[0, 1].set_xlim(0, 105)

# ── Panel C: Payment type mix by trader type (stacked bar) ──────────────────
pay_cols = {'d39': 'Payment on delivery', 'd40': 'Advance to traders', 'd41': 'Advance to farmers'}
pay_by_type = (
    df_cr[paddy_mask]
    .dropna(subset=['trader_type'])
    .groupby('trader_type')[list(pay_cols.keys())]
    .mean()
    .rename(columns=pay_cols)
)
pay_by_type.plot(kind='barh', stacked=False, ax=axes[1, 0],
                  color=['#2196F3', '#FF9800', '#4CAF50'])
axes[1, 0].set_xlabel('Mean % of purchases')
axes[1, 0].set_title('C. Payment arrangement mix by trader type (paddy)')
axes[1, 0].legend(loc='lower right', fontsize=8)

# ── Panel D: Payment type mix by season ─────────────────────────────────────
pay_by_season = (
    df_cr[paddy_mask]
    .dropna(subset=['season'])
    .groupby('season')[list(pay_cols.keys())]
    .mean()
    .rename(columns=pay_cols)
)
pay_by_season.plot(kind='barh', stacked=False, ax=axes[1, 1],
                    color=['#2196F3', '#FF9800', '#4CAF50'])
axes[1, 1].set_xlabel('Mean % of purchases')
axes[1, 1].set_title('D. Payment arrangement mix by season (paddy)')
axes[1, 1].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

---
## Regression Analysis — OLS and IV

**Specification:** OLS of log paddy procurement price on credit arrangement indicators, controlling for upazila FE (absorbs local price levels), season FE, and month FE (absorbs national within-season trends). Standard errors clustered at upazila level.

**Sources merged:**
- `traders_modD_D5.dta` — monthly average paddy procurement price (BDT/kg), trader × season × month
- `traders_modD_D9-D10.dta` — credit arrangement variables at trader × season level
- `traders_modD_D3.dta` — dominant paddy variety per trader-season (Spec 4)

**Specifications:**
1. Binary `any_credit` + upazila + season + month FEs
2. Credit scope dummies (`some_credit`, `all_credit`) + same FEs
3. Continuous payment share variables (`share_adv_farmers`, `share_adv_traders`) + same FEs
4. Spec 1 + paddy variety FEs

**IV:** Leave-one-out upazila-season credit share instruments individual credit receipt. District FEs replace upazila FEs so the instrument retains variation.

In [ ]:
# ── Build regression dataset ─────────────────────────────────────────────────
d9_reg = pd.read_stata(
    trader_file('traders_modD_D9-D10.dta'),
    convert_categoricals=False
)
d5_reg = pd.read_stata(
    trader_file('traders_modD_D5.dta'),
    convert_categoricals=False
)
d3_reg = pd.read_stata(
    trader_file('traders_modD_D3.dta'),
    convert_categoricals=False
)

# Credit indicators at trader × season level
d9_reg['any_credit']        = d9_reg['d35'].map({1: 0, 2: 1, 3: 1})
d9_reg['some_credit']       = (d9_reg['d35'] == 2).astype(float)
d9_reg['all_credit']        = (d9_reg['d35'] == 3).astype(float)
# Payment shares: fill NaN with 0 (no advance reported = 0%)
d9_reg['share_adv_farmers'] = d9_reg['d41'].fillna(0)
d9_reg['share_adv_traders'] = d9_reg['d40'].fillna(0)
d9_reg['credit_days']       = d9_reg['d36']
d9_reg['price_diff_bdt']    = d9_reg['d38']

d9_slim = d9_reg[['tempid', 'season', 'a04', 'a05', 'b09', 'b10',
                   'any_credit', 'some_credit', 'all_credit',
                   'share_adv_farmers', 'share_adv_traders',
                   'credit_days', 'price_diff_bdt']].copy()

# Dominant paddy variety per trader × season (by quantity; ties -> lowest variety code)
d3_main = (d3_reg.sort_values(['pprocqty', 'pproctype'], ascending=[False, True])
                  .groupby(['tempid', 'season'])['pproctype']
                  .first()
                  .reset_index()
                  .rename(columns={'pproctype': 'main_variety'}))

# Merge: outcome (monthly price) + credit regressors + variety
df_r = d5_reg.merge(d9_slim, on=['tempid', 'season'], how='inner', suffixes=('', '_cr'))
df_r = df_r.merge(d3_main, on=['tempid', 'season'], how='left')

# Restrict to paddy traders; drop price outliers (< PRICE_MIN or > PRICE_MAX BDT/kg)
paddy_ok = df_r['b09'].isin([1, 3])
price_ok  = (df_r['pprocmprice'] > PRICE_MIN) & (df_r['pprocmprice'] < PRICE_MAX)
df_r = df_r[paddy_ok & price_ok].copy()

df_r['log_price']    = np.log(df_r['pprocmprice'])
df_r['upazila_str']  = df_r['a05'].astype(str)
df_r['district_str'] = df_r['a04'].astype(str)
df_r['season_str']   = df_r['season'].astype(str)
df_r['month_str']    = df_r['month'].astype(str)
df_r['variety_str']  = df_r['main_variety'].fillna(-1).astype(str)

df_r = df_r.dropna(subset=['any_credit', 'log_price']).copy()

print(f"Regression dataset: N={len(df_r):,} trader-season-month obs")
print(f"Unique traders    : {df_r['tempid'].nunique():,}")
print(f"Unique upazilas   : {df_r['upazila_str'].nunique()}")
print(f"log_price — mean: {df_r['log_price'].mean():.3f}, "
      f"std: {df_r['log_price'].std():.3f} (raw: {df_r['pprocmprice'].mean():.2f} BDT/kg)")

In [ ]:
# ── OLS Regressions: Specs 1–4 ───────────────────────────────────────────────
# All specs use upazila + season + month fixed effects.
# Standard errors clustered at the upazila level.

FE = '+ C(season_str) + C(month_str) + C(upazila_str)'
kw  = lambda df: {'cov_type': 'cluster', 'cov_kwds': {'groups': df['upazila_str']}}

# Spec 1: binary any_credit
m1 = smf.ols(f'log_price ~ any_credit {FE}', data=df_r).fit(**kw(df_r))

# Spec 2: credit scope dummies (omitted = no credit)
m2 = smf.ols(f'log_price ~ some_credit + all_credit {FE}', data=df_r).fit(**kw(df_r))

# Spec 3: continuous payment shares + binary credit flag
m3 = smf.ols(
    f'log_price ~ any_credit + share_adv_farmers + share_adv_traders {FE}',
    data=df_r
).fit(**kw(df_r))

# Spec 4: Spec 1 + paddy variety FEs
m4 = smf.ols(
    f'log_price ~ any_credit + C(variety_str) {FE}',
    data=df_r
).fit(**kw(df_r))

# ── Results table ─────────────────────────────────────────────────────────────
def reg_row(m, var, name=None):
    name = name or var
    if var not in m.params:
        return f'  {name:40s}   —'
    c = m.params[var]; se = m.bse[var]; p = m.pvalues[var]
    stars = '***' if p < .01 else ('**' if p < .05 else ('*' if p < .1 else ''))
    return f'  {name:40s}  {c:+.4f}  ({se:.4f})  {stars}'

header = f"{'Variable':42s}  {'(1)':>18}  {'(2)':>18}  {'(3)':>18}  {'(4)':>18}"
print('\nTable: OLS — Dependent variable: log(paddy procurement price, BDT/kg)')
print('─' * 80)
print(header)
print('─' * 80)

for m, var, label in [
    (m1, 'any_credit',        'Any credit from suppliers'),
    (m2, 'some_credit',       '  Some suppliers'),
    (m2, 'all_credit',        '  All suppliers'),
    (m3, 'any_credit',        'Any credit (Spec 3)'),
    (m3, 'share_adv_farmers', '  Adv. to farmers (% of purch.)'),
    (m3, 'share_adv_traders', '  Adv. to traders (% of purch.)'),
]:
    models = {'(1)': m1, '(2)': m2, '(3)': m3, '(4)': m4}
    row = f'  {label:40s}'
    for tag, mod in models.items():
        if var in mod.params:
            c = mod.params[var]; se = mod.bse[var]; p = mod.pvalues[var]
            stars = '***' if p<.01 else ('**' if p<.05 else ('*' if p<.1 else '   '))
            row += f'  {c:+.4f}{stars} ({se:.4f})'
        else:
            row += f'  {"—":>18}'
    print(row)

print('─' * 80)
for tag, m in [('(1)', m1), ('(2)', m2), ('(3)', m3), ('(4)', m4)]:
    pass
print(f"  {'N':40s}  {int(m1.nobs):>18,}  {int(m2.nobs):>18,}  {int(m3.nobs):>18,}  {int(m4.nobs):>18,}")
print(f"  {'R²':40s}  {m1.rsquared:>18.3f}  {m2.rsquared:>18.3f}  {m3.rsquared:>18.3f}  {m4.rsquared:>18.3f}")
print(f"  {'Upazila FEs':40s}  {'Yes':>18}  {'Yes':>18}  {'Yes':>18}  {'Yes':>18}")
print(f"  {'Season + Month FEs':40s}  {'Yes':>18}  {'Yes':>18}  {'Yes':>18}  {'Yes':>18}")
print(f"  {'Variety FEs':40s}  {'No':>18}  {'No':>18}  {'No':>18}  {'Yes':>18}")
print('─' * 80)
print('Notes: SE clustered at upazila level. *** p<0.01, ** p<0.05, * p<0.1.')

In [ ]:
# ── Implicit interest rate from price premium variable (D38/D36) ─────────────
# D37 = whether trader paid a different price when buying on credit
# D38 = price difference (BDT/kg) — positive = higher price paid for credit
# D36 = credit duration (days)
# Implied annual rate = (price_diff / price_level) * (365 / credit_days) * 100

# Merge D9-D10 price-premium variables with average price from D5
avg_price = (d5_reg[(d5_reg['pprocmprice'] > PRICE_MIN) & (d5_reg['pprocmprice'] < PRICE_MAX)]
             .groupby(['tempid', 'season'])['pprocmprice']
             .mean()
             .reset_index()
             .rename(columns={'pprocmprice': 'avg_price'}))

impl = (d9_reg[d9_reg['d37'] == 1]          # traders paying a price premium for credit
        .dropna(subset=['price_diff_bdt', 'credit_days'])
        .merge(avg_price, on=['tempid', 'season'], how='inner'))
impl = impl[(impl['avg_price'] > PRICE_MIN) & (impl['avg_price'] < PRICE_MAX)]

impl['implied_annual_rate_pct'] = (
    impl['price_diff_bdt'] / impl['avg_price']
) * (365 / impl['credit_days']) * 100

print(f"Obs. with explicit price premium (D37=Yes): {len(impl)}")
print(f"\nImplied annual interest rate (%):")
print(impl['implied_annual_rate_pct'].describe().round(1))
print(f"\nMedian credit duration among premium payers: {impl['credit_days'].median():.0f} days")
print(f"Median price difference: {impl['price_diff_bdt'].median():.3f} BDT/kg")
print(f"Median avg procurement price: {impl['avg_price'].median():.2f} BDT/kg")

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
impl['implied_annual_rate_pct'].clip(upper=200).hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(impl['implied_annual_rate_pct'].median(), color='red', linestyle='--', label=f"Median={impl['implied_annual_rate_pct'].median():.0f}%")
axes[0].set_xlabel('Implied annual interest rate (%)')
axes[0].set_title('Distribution of implicit interest rates\n(from D38/D36, price-premium payers only)')
axes[0].legend()

axes[1].scatter(impl['credit_days'], impl['implied_annual_rate_pct'].clip(upper=200),
                alpha=0.6, color='coral')
axes[1].set_xlabel('Credit duration (days)')
axes[1].set_ylabel('Implied annual rate (%)')
axes[1].set_title('Interest rate vs. credit duration')
plt.tight_layout()
plt.show()

In [ ]:
# ── IV Robustness: 2SLS with leave-one-out upazila-season credit share ────────
# Rationale: credit receipt is potentially endogenous — traders with better
# bargaining power may both receive more credit AND pay lower prices.
# Instrument: leave-one-out average credit rate in the same upazila-season
# (excludes own observation to avoid mechanical correlation).
# District FEs replace upazila FEs so the instrument retains cross-upazila variation.

# Build LOO instrument
loo_mean = df_r.groupby(['upazila_str', 'season_str'])['any_credit'].transform(
    lambda x: (x.sum() - x) / (x.count() - 1)
)
df_r = df_r.copy()
df_r['loo_credit_rate'] = loo_mean
df_iv = df_r.dropna(subset=['loo_credit_rate']).copy()

# First stage: regress any_credit on instrument + district + season + month FEs
fs = smf.ols(
    'any_credit ~ loo_credit_rate + C(district_str) + C(season_str) + C(month_str)',
    data=df_iv
).fit()
fstat_instrument = (fs.params['loo_credit_rate'] / fs.bse['loo_credit_rate']) ** 2
print(f"First stage:")
print(f"  Instrument coef: {fs.params['loo_credit_rate']:.4f}  "
      f"(se={fs.bse['loo_credit_rate']:.4f})")
print(f"  F-statistic on instrument: {fstat_instrument:.1f}  "
      f"(rule-of-thumb threshold: 10)")

# OLS baseline with district FEs (for apples-to-apples comparison with IV)
ols_d = smf.ols(
    'log_price ~ any_credit + C(district_str) + C(season_str) + C(month_str)',
    data=df_iv
).fit(cov_type='cluster', cov_kwds={'groups': df_iv['upazila_str']})

# 2SLS via linearmodels
df_iv['const'] = 1.0
dummies = pd.concat([
    pd.get_dummies(df_iv['district_str'], drop_first=True, prefix='dist').astype(float),
    pd.get_dummies(df_iv['season_str'],   drop_first=True, prefix='s').astype(float),
    pd.get_dummies(df_iv['month_str'],    drop_first=True, prefix='m').astype(float),
], axis=1)
exog  = pd.concat([df_iv[['const']], dummies], axis=1)
endog = df_iv[['any_credit']]
instr = df_iv[['loo_credit_rate']]
y_iv  = df_iv['log_price']

iv_res = IV2SLS(y_iv, exog, endog, instr).fit(
    cov_type='clustered', clusters=df_iv['upazila_str']
)

c_iv = iv_res.params['any_credit']
se_iv = iv_res.std_errors['any_credit']
p_iv  = iv_res.pvalues['any_credit']
c_ols = ols_d.params['any_credit']
se_ols = ols_d.bse['any_credit']
p_ols  = ols_d.pvalues['any_credit']

def pstars(p):
    return '***' if p<.01 else ('**' if p<.05 else ('*' if p<.1 else ''))

print(f"\nIV Robustness (district FEs, N={int(iv_res.nobs):,}):")
print(f"{'':4s}{'Method':20s} {'Coef':>10} {'SE':>10} {'p':>8}")
print(f"{'':4s}{'OLS (district FE)':20s} {c_ols:>+10.4f} {se_ols:>10.4f} {p_ols:>8.3f} {pstars(p_ols)}")
print(f"{'':4s}{'2SLS (district FE)':20s} {c_iv:>+10.4f}  {se_iv:>10.4f} {p_iv:>8.3f} {pstars(p_iv)}")
print()
print("Note: OLS (upazila FE, Spec 1):", f"{m1.params['any_credit']:+.4f}",
      f"({m1.bse['any_credit']:.4f})", pstars(m1.pvalues['any_credit']) or "n.s.")

---
## Heterogeneity Analysis

Three tests to distinguish *competitive equilibrium* (null holds everywhere)
from *power asymmetry* (null holds only in thick/competitive markets):

1. **Local credit intensity** — interaction of `any_credit` with the upazila-season credit share, plus the
   *mixed-market* subsample (upazila-seasons with credit share < 1), where within-market variation in credit receipt exists
2. **Trader type** — separate credit effects for faria/bepari vs aratdar vs broker
3. **Price variance** — does credit reduce within-season price dispersion even if not levels?

In [ ]:
# ── H1: Credit effect × local credit market intensity ────────────────────────
# District FEs (consistent with IV spec) so upazila-level credit share
# retains cross-upazila identifying variation within districts.

upz_share = (
    df_r.groupby(['upazila_str', 'season_str'])['any_credit']
    .mean().reset_index()
    .rename(columns={'any_credit': 'upz_cr_share'})
)
df_h = df_r.merge(upz_share, on=['upazila_str', 'season_str'], how='left').copy()
share_mean = df_h['upz_cr_share'].mean()
share_med  = df_h['upz_cr_share'].median()
df_h['upz_cr_share_c'] = df_h['upz_cr_share'] - share_mean

FE_d = '+ C(district_str) + C(season_str) + C(month_str)'
kw_h = {'cov_type': 'cluster', 'cov_kwds': {'groups': df_h['upazila_str']}}

# Baseline (district FEs)
m_h0 = smf.ols(f'log_price ~ any_credit {FE_d}', data=df_h).fit(**kw_h)

# Interaction: credit effect moderated by local share (mean-centred)
m_h1 = smf.ols(f'log_price ~ any_credit * upz_cr_share_c {FE_d}', data=df_h).fit(**kw_h)

# The median share is 1.0 (most markets are universal), so a median split is
# degenerate. Instead use mixed markets (share < 1.0), where within-market
# variation in credit receipt actually exists.
pct_universal = (df_h['upz_cr_share'] == 1.0).mean() * 100
df_mixed = df_h[df_h['upz_cr_share'] < 1.0].copy()
m_h1_mixed = smf.ols(f'log_price ~ any_credit {FE_d}', data=df_mixed).fit(
    cov_type='cluster', cov_kwds={'groups': df_mixed['upazila_str']})

print(f'Share mean={share_mean:.3f}  median={share_med:.3f}')
print(f'Universal markets (share=1.0): {pct_universal:.1f}% of obs')
print()
print('Interaction model:')
for v in ['any_credit', 'upz_cr_share_c', 'any_credit:upz_cr_share_c']:
    if v in m_h1.params:
        c, se, p = m_h1.params[v], m_h1.bse[v], m_h1.pvalues[v]
        star = '***' if p<.01 else ('**' if p<.05 else ('*' if p<.1 else ''))
        print(f'  {v:42s} {c:+.4f} ({se:.4f}) {star}')
c, se, p = m_h1_mixed.params['any_credit'], m_h1_mixed.bse['any_credit'], m_h1_mixed.pvalues['any_credit']
print(f'\nMixed-market subsample (share<1.0): {c:+.4f} ({se:.4f})  p={p:.3f}  N={int(m_h1_mixed.nobs):,}')

In [ ]:
# ── H2: Heterogeneity by trader type ─────────────────────────────────────────
# Ref category: Faria/bepari (b10==1); upazila FEs

df_h['is_aratdar'] = (df_h['b10'] == 2).astype(float)
df_h['is_broker']  = (df_h['b10'] == 3).astype(float)

FE_u = '+ C(season_str) + C(month_str) + C(upazila_str)'
kw_u = {'cov_type': 'cluster', 'cov_kwds': {'groups': df_h['upazila_str']}}

m_h2 = smf.ols(
    'log_price ~ any_credit + is_aratdar + is_broker'
    ' + any_credit:is_aratdar + any_credit:is_broker' + FE_u,
    data=df_h
).fit(**kw_u)

print('Trader type heterogeneity (ref = Faria/bepari):')
for v in ['any_credit', 'is_aratdar', 'is_broker',
          'any_credit:is_aratdar', 'any_credit:is_broker']:
    if v in m_h2.params:
        c, se, p = m_h2.params[v], m_h2.bse[v], m_h2.pvalues[v]
        star = '***' if p<.01 else ('**' if p<.05 else ('*' if p<.1 else ''))
        print(f'  {v:42s} {c:+.4f} ({se:.4f}) {star}')

c_faria  = m_h2.params['any_credit']
c_arat   = m_h2.params['any_credit'] + m_h2.params.get('any_credit:is_aratdar', 0)
c_broker = m_h2.params['any_credit'] + m_h2.params.get('any_credit:is_broker', 0)
print(f'\nTotal credit effect by type:')
print(f'  Faria/bepari : {c_faria:+.4f}')
print(f'  Aratdar      : {c_arat:+.4f}')
print(f'  Broker       : {c_broker:+.4f}')

In [ ]:
# ── H3: Price variance — does credit dampen within-season price dispersion? ───

pv = (
    df_r.groupby(['tempid', 'season_str', 'upazila_str',
                  'any_credit', 'some_credit', 'all_credit'])
    ['pprocmprice']
    .agg(price_sd='std', price_mean='mean', n_months='count')
    .reset_index()
)
pv = pv[pv['n_months'] >= 2].copy()
pv['price_cv'] = pv['price_sd'] / pv['price_mean'] * 100

kw_pv = {'cov_type': 'cluster', 'cov_kwds': {'groups': pv['upazila_str']}}

m_v1 = smf.ols('price_sd ~ any_credit + C(season_str) + C(upazila_str)',
               data=pv).fit(**kw_pv)
m_v2 = smf.ols('price_cv ~ any_credit + C(season_str) + C(upazila_str)',
               data=pv).fit(**kw_pv)
m_v3 = smf.ols('price_cv ~ some_credit + all_credit + C(season_str) + C(upazila_str)',
               data=pv).fit(**kw_pv)

print(f'Price variance sample: N={len(pv):,} trader-seasons (>= 2 active months)')
print(f'Mean SD={pv["price_sd"].mean():.3f} BDT/kg  Mean CV={pv["price_cv"].mean():.2f}%')
print()
for label, m, vars_ in [
    ('SD  (m_v1)', m_v1, ['any_credit']),
    ('CV  (m_v2)', m_v2, ['any_credit']),
    ('CV scope (m_v3)', m_v3, ['some_credit', 'all_credit'])
]:
    row = f'  {label:20s}'
    for v in vars_:
        c, se, p = m.params[v], m.bse[v], m.pvalues[v]
        star = '***' if p<.01 else ('**' if p<.05 else ('*' if p<.1 else '   '))
        row += f'  {v}: {c:+.4f} ({se:.4f}){star}'
    print(row)

---
## Paper Output — Export tables (.tex) and figures (.pdf) to `paper/`

Run these cells after all analysis cells above. Outputs land in `paper/tables/` and `paper/figures/`,
which are `\input{}`-ed directly by `paper/main.tex`. File numbers match the table numbers in the paper.

In [ ]:
matplotlib.rcParams.update({
    'font.family': 'serif', 'font.size': 10,
    'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'figure.dpi': 300,
})
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── tab2_sumstats.tex ─────────────────────────────────────────────────────────
paddy_cr = df_cr[df_cr['b09'].isin([1, 3])]

n_traders  = df_cr['tempid'].nunique()
n_ts_total = len(df_cr)
n_ts_paddy = len(paddy_cr)
pct_paddy  = (df_cr['b09'] == 1).mean() * 100
pct_rice   = (df_cr['b09'] == 2).mean() * 100
pct_both   = (df_cr['b09'] == 3).mean() * 100
pct_faria  = (paddy_cr['b10'] == 1).mean() * 100
pct_arat   = (paddy_cr['b10'] == 2).mean() * 100
pct_brok   = (paddy_cr['b10'] == 3).mean() * 100
pct_other  = (paddy_cr['b10'] == 4).mean() * 100
pct_any_cr = paddy_cr['any_credit_paddy'].mean() * 100
pct_some   = pct_of_answered(paddy_cr['d35'], 2)
pct_all    = pct_of_answered(paddy_cr['d35'], 3)
mean_days  = paddy_cr['d36'].mean()
mean_adv_f = paddy_cr['d41'].mean()
mean_adv_t = paddy_cr['d40'].mean()
mean_deliv = paddy_cr['d39'].mean()
pct_prem   = paddy_cr['price_premium_paddy'].mean() * 100
mean_price = df_r['pprocmprice'].mean()
sd_price   = df_r['pprocmprice'].std()
n_reg      = len(df_r)

rows = [
    r'\begin{tabular}{lS[table-format=4.1]}',
    r'\toprule',
    r'Variable & \multicolumn{1}{c}{Value} \\',
    r'\midrule',
    r'\multicolumn{2}{l}{\textit{Panel A: Sample characteristics}} \\',
    r'\addlinespace[2pt]',
    f'Unique traders & {n_traders:.0f} \\\\',
    f'Trader-season observations & {n_ts_total:.0f} \\\\',
    r'\quad Paddy traders (\%) & ' + f'{pct_paddy:.1f} \\\\',
    r'\quad Rice traders (\%) & ' + f'{pct_rice:.1f} \\\\',
    r'\quad Both (\%) & ' + f'{pct_both:.1f} \\\\',
    r'\addlinespace[4pt]',
    r'\multicolumn{2}{l}{\textit{Paddy trader types (\% of paddy trader-seasons)}} \\',
    r'\addlinespace[2pt]',
    r'\quad Faria/bepari & ' + f'{pct_faria:.1f} \\\\',
    r'\quad Aratdar & ' + f'{pct_arat:.1f} \\\\',
    r'\quad Broker/intermediary & ' + f'{pct_brok:.1f} \\\\',
    r'\quad Other & ' + f'{pct_other:.1f} \\\\',
    r'\addlinespace[4pt]',
    r'\multicolumn{2}{l}{\textit{Panel B: Credit variables (paddy trader-seasons)}} \\',
    r'\addlinespace[2pt]',
    f'Any supplier credit (\\%) & {pct_any_cr:.1f} \\\\',
    r'\quad Some suppliers (\%) & ' + f'{pct_some:.1f} \\\\',
    r'\quad All suppliers (\%) & ' + f'{pct_all:.1f} \\\\',
    f'Mean credit duration (days) & {mean_days:.1f} \\\\',
    f'Mean advance-to-farmers (\\% of purchases) & {mean_adv_f:.1f} \\\\',
    f'Mean advance-to-traders (\\% of purchases) & {mean_adv_t:.1f} \\\\',
    f'Mean payment on delivery (\\% of purchases) & {mean_deliv:.1f} \\\\',
    f'Explicit price premium reported (\\%) & {pct_prem:.1f} \\\\',
    r'\addlinespace[4pt]',
    r'\multicolumn{2}{l}{\textit{Panel C: Price variable (trader-season-month obs.)}} \\',
    r'\addlinespace[2pt]',
    f'Regression sample obs. & {n_reg:.0f} \\\\',
    f'Mean procurement price (BDT/kg) & {mean_price:.2f} \\\\',
    f'SD procurement price (BDT/kg) & {sd_price:.2f} \\\\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(f'{TABLES}/tab2_sumstats.tex', rows)
print('tab2_sumstats done.')

In [ ]:
# ── tab3_credit_by_type.tex ───────────────────────────────────────────────────
type_labels = {1: 'Faria/bepari', 2: 'Aratdar', 3: 'Broker/intermediary', 4: 'Other'}

rows = [
    r'\begin{tabular}{lcccccc}',
    r'\toprule',
    r' & \multicolumn{3}{c}{Credit scope (\%)} & \multicolumn{3}{c}{Payment mix (mean \%)} \\',
    r'\cmidrule(lr){2-4}\cmidrule(lr){5-7}',
    r'Trader type & None & Some & All & On delivery & Adv.\ to traders & Adv.\ to farmers \\',
    r'\midrule',
]
for code, label in type_labels.items():
    sub = paddy_cr[paddy_cr['b10'] == code]
    if len(sub) == 0:
        continue
    none_ = pct_of_answered(sub['d35'], 1)
    some_ = pct_of_answered(sub['d35'], 2)
    all_  = pct_of_answered(sub['d35'], 3)
    deliv = sub['d39'].mean()
    adv_t = sub['d40'].mean()
    adv_f = sub['d41'].mean()
    rows.append(
        f'{label} & {none_:.1f} & {some_:.1f} & {all_:.1f}'
        f' & {deliv:.1f} & {adv_t:.1f} & {adv_f:.1f} \\\\'
    )

# Chi-square test: is credit scope (None/Some/All) independent of trader type?
q = paddy_cr.dropna(subset=['d35', 'b10'])
ct = pd.crosstab(q['b10'], q['d35'])
chi2_stat, chi2_p, chi2_df, chi2_exp = chi2_contingency(ct)
# Permutation p-value: robust to the small expected counts in the broker row
xi = pd.factorize(q['b10'])[0]
yi = pd.factorize(q['d35'])[0]
R, C = xi.max() + 1, yi.max() + 1

def chi2_of(y):
    obs = np.bincount(xi * C + y, minlength=R * C).reshape(R, C)
    exp = obs.sum(1, keepdims=True) * obs.sum(0, keepdims=True) / obs.sum()
    return ((obs - exp) ** 2 / exp).sum()

rng = np.random.default_rng(2018)
perm = np.array([chi2_of(rng.permutation(yi)) for _ in range(10_000)])
chi2_perm_p = (1 + (perm >= chi2_stat).sum()) / (1 + len(perm))
print(f'chi2({chi2_df}) = {chi2_stat:.1f}  asymptotic p = {chi2_p:.2g}  permutation p = {chi2_perm_p:.2g}  '
      f'(expected counts < 5: {(chi2_exp < 5).sum()} of {chi2_exp.size} cells)')

# Overall row
none_a = pct_of_answered(paddy_cr['d35'], 1)
some_a = pct_of_answered(paddy_cr['d35'], 2)
all_a  = pct_of_answered(paddy_cr['d35'], 3)
d_a = paddy_cr['d39'].mean(); at_a = paddy_cr['d40'].mean(); af_a = paddy_cr['d41'].mean()
rows += [
    r'\midrule',
    f'\\textit{{All traders}} & {none_a:.1f} & {some_a:.1f} & {all_a:.1f}'
    f' & {d_a:.1f} & {at_a:.1f} & {af_a:.1f} \\\\',
    r'\addlinespace[2pt]',
    rf'$\chi^2$({chi2_df}), credit scope $\times$ trader type & '
    rf'\multicolumn{{3}}{{c}}{{{fmt_c(chi2_stat, chi2_perm_p, 1)}}} & & & \\',
    r'\addlinespace[2pt]',
    r'\multicolumn{7}{p{0.9\textwidth}}{\footnotesize \textit{Note:} Paddy/both trader-seasons only. '
    r'Credit scope: percent of trader-seasons answering D35. '
    r'Payment mix columns: means among trader-seasons with any supplier credit. '
    r'$\chi^2$: Pearson test of independence between trader type and credit scope; '
    r'stars use a permutation $p$-value (10,000 draws), robust to small expected counts.} \\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(f'{TABLES}/tab3_credit_by_type.tex', rows)
print('tab3_credit_by_type done.')

In [ ]:
# ── tab4_credit_by_season.tex ─────────────────────────────────────────────────
season_order = ['Aman 2016/17', 'Boro 2017', 'Aus 2017']

rows = [
    r'\begin{tabular}{lcccccc}',
    r'\toprule',
    r' & \multicolumn{3}{c}{Credit scope (\%)} & \multicolumn{3}{c}{Payment mix (mean \%)} \\',
    r'\cmidrule(lr){2-4}\cmidrule(lr){5-7}',
    r'Season & None & Some & All & On delivery & Adv.\ to traders & Adv.\ to farmers \\',
    r'\midrule',
]
for label in season_order:
    sub = paddy_cr[paddy_cr['season'] == label]
    if len(sub) == 0:
        continue
    none_ = pct_of_answered(sub['d35'], 1)
    some_ = pct_of_answered(sub['d35'], 2)
    all_  = pct_of_answered(sub['d35'], 3)
    deliv = sub['d39'].mean()
    adv_t = sub['d40'].mean()
    adv_f = sub['d41'].mean()
    rows.append(
        f'{label} & {none_:.1f} & {some_:.1f} & {all_:.1f}'
        f' & {deliv:.1f} & {adv_t:.1f} & {adv_f:.1f} \\\\'
    )

rows += [
    r'\midrule',
    f'\\textit{{All seasons}} & {none_a:.1f} & {some_a:.1f} & {all_a:.1f}'
    f' & {d_a:.1f} & {at_a:.1f} & {af_a:.1f} \\\\',
    r'\addlinespace[2pt]',
    r'\multicolumn{7}{p{0.9\textwidth}}{\footnotesize \textit{Note:} Paddy/both trader-seasons only.} \\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(f'{TABLES}/tab4_credit_by_season.tex', rows)
print('tab4_credit_by_season done.')

In [ ]:
# ── tab5_credit_by_district.tex ───────────────────────────────────────────────
# Build district-level table with N, % any credit, mean days, mean payment shares
dist_tbl = (
    paddy_cr.dropna(subset=['any_credit_paddy', 'district'])
    .groupby('district')
    .agg(
        N=('any_credit_paddy', 'count'),
        n_traders=('tempid', 'nunique'),
        pct_any=('any_credit_paddy', lambda x: x.mean() * 100),
        mean_days=('d36', 'mean'),
        mean_deliv=('d39', 'mean'),
        mean_adv_t=('d40', 'mean'),
        mean_adv_f=('d41', 'mean'),
    )
    .round(1)
    .sort_index()
    .sort_values('pct_any', ascending=False, kind='stable')
)

# Confidentiality: drop districts with fewer than MIN_TRADERS distinct traders
n_dropped = int((dist_tbl['n_traders'] < MIN_TRADERS).sum())
dist_tbl = dist_tbl[dist_tbl['n_traders'] >= MIN_TRADERS]
print(f'Districts suppressed (< {MIN_TRADERS} traders): {n_dropped}')

# Top 10 and bottom 10 by credit prevalence
top10    = dist_tbl.head(10)
bottom10 = dist_tbl.tail(10)

def dist_block(df, caption):
    out = [f'\\multicolumn{{7}}{{l}}{{\\textit{{{caption}}}}} \\\\', r'\addlinespace[2pt]']
    for dist, row in df.iterrows():
        out.append(
            f'{dist} & {int(row["N"])} & {row["pct_any"]:.1f}'
            f' & {row["mean_days"]:.1f} & {row["mean_deliv"]:.1f}'
            f' & {row["mean_adv_t"]:.1f} & {row["mean_adv_f"]:.1f} \\\\'
        )
    return out

rows = [
    r'\begin{tabular}{lcccccc}',
    r'\toprule',
    r'District & N & Any credit & Credit & On & Adv.\ to & Adv.\ to \\',
    r' & & (\%) & days & delivery & traders & farmers \\',
    r'\midrule',
]
rows += dist_block(top10,    'Highest credit prevalence (top 10 districts)')
rows.append(r'\addlinespace[4pt]')
rows += dist_block(bottom10, 'Lowest credit prevalence (bottom 10 districts)')
rows += [
    r'\addlinespace[2pt]',
    r'\multicolumn{7}{p{0.9\textwidth}}{\footnotesize \textit{Note:} Paddy/both trader-seasons only. '
    r'Credit days and payment shares: means among trader-seasons with any supplier credit. '
    r'Districts with fewer than ' + str(MIN_TRADERS) + r' sampled traders are omitted.} \\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(f'{TABLES}/tab5_credit_by_district.tex', rows)
print('tab5_credit_by_district done.')

In [ ]:
# ── tab7_ols.tex ──────────────────────────────────────────────────────────────
# Specs 1-4: OLS with upazila + season + month FEs

var_labels = {
    'any_credit':        'Any supplier credit',
    'some_credit':       r'\quad Some suppliers',
    'all_credit':        r'\quad All suppliers',
    'share_adv_farmers': r'\quad Adv.\ to farmers (\% purch.)',
    'share_adv_traders': r'\quad Adv.\ to traders (\% purch.)',
}
models = [m1, m2, m3, m4]
spec_labels = ['(1)', '(2)', '(3)', '(4)']

rows = [
    r'\begin{tabular}{lcccc}',
    r'\toprule',
    r'& (1) & (2) & (3) & (4) \\',
    r'\midrule',
]

for var, label in var_labels.items():
    coef_parts = []
    se_parts   = []
    for m in models:
        if var in m.params:
            coef_parts.append(fmt_c(m.params[var], m.pvalues[var]))
            se_parts.append(fmt_se(m.bse[var]))
        else:
            coef_parts.append('')
            se_parts.append('')
    rows.append(label + ' & ' + ' & '.join(coef_parts) + r' \\')
    rows.append(r'& ' + ' & '.join(se_parts) + r' \\')
    rows.append(r'\addlinespace[2pt]')

# Footer statistics
rows += [r'\midrule']
rows.append(r'Observations & ' + ' & '.join(f'{int(m.nobs):,}' for m in models) + r' \\')
rows.append(r'$R^2$ & ' + ' & '.join(f'{m.rsquared:.3f}' for m in models) + r' \\')
rows.append(r'Upazila FEs & Yes & Yes & Yes & Yes \\')
rows.append(r'Season \& month FEs & Yes & Yes & Yes & Yes \\')
rows.append(r'Variety FEs & No & No & No & Yes \\')
rows += [
    r'\addlinespace[2pt]',
    r'\multicolumn{5}{p{0.85\textwidth}}{\footnotesize \textit{Notes:} '
    r'Dependent variable: log paddy procurement price (BDT/kg). '
    r'Standard errors (in parentheses) clustered at the upazila level. '
    r'Sample restricted to paddy/both trader-season-months with price between 5 and 50 BDT/kg. '
    r'$^{*}p<0.10$, $^{**}p<0.05$, $^{***}p<0.01$.} \\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(f'{TABLES}/tab7_ols.tex', rows)
print('tab7_ols done.')

In [ ]:
# ── tab8_iv.tex ───────────────────────────────────────────────────────────────
# Three-column: OLS upazila FE | OLS district FE | 2SLS district FE

def iv_fmt_c(coef, pval):
    return fmt_c(coef, pval)

def iv_fmt_se(se):
    return f'({se:.4f})'

# Pull coefficients from each spec
ols_up_c  = m1.params['any_credit'];   ols_up_se  = m1.bse['any_credit'];   ols_up_p  = m1.pvalues['any_credit']
ols_di_c  = ols_d.params['any_credit']; ols_di_se = ols_d.bse['any_credit']; ols_di_p  = ols_d.pvalues['any_credit']
tsls_c    = iv_res.params['any_credit']; tsls_se   = iv_res.std_errors['any_credit']; tsls_p = iv_res.pvalues['any_credit']

rows = [
    r'\begin{tabular}{lccc}',
    r'\toprule',
    r'& \multicolumn{2}{c}{OLS} & 2SLS \\',
    r'\cmidrule(lr){2-3}',
    r'& Upazila FE & District FE & District FE \\',
    r'& (1) & (2) & (3) \\',
    r'\midrule',
    r'Any supplier credit & '
        + iv_fmt_c(ols_up_c, ols_up_p) + ' & '
        + iv_fmt_c(ols_di_c, ols_di_p) + ' & '
        + iv_fmt_c(tsls_c,   tsls_p)   + r' \\',
    r'& '
        + iv_fmt_se(ols_up_se) + ' & '
        + iv_fmt_se(ols_di_se) + ' & '
        + iv_fmt_se(tsls_se)   + r' \\',
    r'\addlinespace[4pt]',
    r'\midrule',
    f'Observations & {int(m1.nobs):,} & {int(ols_d.nobs):,} & {int(iv_res.nobs):,} \\\\',
    r'Upazila FEs & Yes & No & No \\',
    r'District FEs & No & Yes & Yes \\',
    r'Season \& month FEs & Yes & Yes & Yes \\',
    r'First-stage $F$ & & & ' + f'{fstat_instrument:.1f}' + r' \\',
    r'\addlinespace[2pt]',
    r'\multicolumn{4}{p{0.75\textwidth}}{\footnotesize \textit{Notes:} '
    r'Dependent variable: log paddy procurement price. '
    r'Instrument (col.\ 3): leave-one-out upazila-season credit share. '
    r'SE clustered at upazila level. '
    r'$^{*}p<0.10$, $^{**}p<0.05$, $^{***}p<0.01$.} \\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(f'{TABLES}/tab8_iv.tex', rows)
print('tab8_iv done.')

In [ ]:
# ── tab9_heterogeneity.tex ───────────────────────────────────────────────────
# Panels: (A) credit share interaction, (B) mixed-market subsample, (C) trader type

def row_pair(label, m, var):
    if var not in m.params:
        return [label + r' & \\', r'& \\']
    c, se, p = m.params[var], m.bse[var], m.pvalues[var]
    return [label + ' & ' + fmt_c(c, p) + r' \\', r'& ' + fmt_se(se) + r' \\']

rows = [
    r'\begin{tabular}{lc}',
    r'\toprule',
    r'& Log price \\',
    r'\midrule',
    r'\multicolumn{2}{l}{\textit{Panel A: Interaction with local credit market intensity (district FEs)}} \\',
    r'\addlinespace[2pt]',
]
rows += row_pair('Any supplier credit (at mean local share)', m_h1, 'any_credit')
rows += row_pair('Local credit share (mean-centred)', m_h1, 'upz_cr_share_c')
rows += row_pair(r'\quad Credit $\times$ local share', m_h1, 'any_credit:upz_cr_share_c')
rows += [
    r'\addlinespace[2pt]',
    f'Observations & {int(m_h1.nobs):,} \\\\',
    f'$R^2$ & {m_h1.rsquared:.3f} \\\\',
    r'\addlinespace[6pt]',
    r'\multicolumn{2}{l}{\textit{Panel B: Mixed-market subsample (local credit share $<$ 100\%, district FEs)}} \\',
    r'\addlinespace[2pt]',
]
c_mix, se_mix, p_mix = (m_h1_mixed.params['any_credit'], m_h1_mixed.bse['any_credit'],
                        m_h1_mixed.pvalues['any_credit'])
rows += [
    f'Any supplier credit (mixed-market upazila-seasons) & {fmt_c(c_mix, p_mix)} \\\\',
    r'& ' + fmt_se(se_mix) + r' \\',
    r'\addlinespace[2pt]',
    f'Observations & {int(m_h1_mixed.nobs):,} \\\\',
    r'\addlinespace[6pt]',
    r'\multicolumn{2}{l}{\textit{Panel C: Heterogeneity by trader type (upazila FEs; ref.\ = Faria/bepari)}} \\',
    r'\addlinespace[2pt]',
]
for label, var in [('Any credit', 'any_credit'),
                   (r'\quad $\times$ Aratdar', 'any_credit:is_aratdar'),
                   (r'\quad $\times$ Broker/intermediary', 'any_credit:is_broker')]:
    rows += row_pair(label, m_h2, var)
rows += [
    r'\addlinespace[2pt]',
    f'Observations & {int(m_h2.nobs):,} \\\\',
    r'\addlinespace[2pt]',
    r'\multicolumn{2}{p{0.7\textwidth}}{\footnotesize \textit{Notes:}'
    r' Panels A and B use district FEs; Panel C uses upazila FEs.'
    r' Standard errors clustered at upazila level.'
    r' $^{*}p<0.10$, $^{**}p<0.05$, $^{***}p<0.01$.} \\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(TABLES / 'tab9_heterogeneity.tex', rows)

In [ ]:
# ── tab10_price_variance.tex ─────────────────────────────────────────────────
rows = [
    r'\begin{tabular}{lccc}',
    r'\toprule',
    r'& Price SD & Price CV & Price CV \\',
    r'& (BDT/kg) & (\%) & (\%) \\',
    r'& (1) & (2) & (3) \\',
    r'\midrule',
]
var_rows = [
    ('Any supplier credit',   'any_credit',  [m_v1, m_v2, None]),
    (r'\quad Some suppliers', 'some_credit', [None, None, m_v3]),
    (r'\quad All suppliers',  'all_credit',  [None, None, m_v3]),
]
for label, var, models in var_rows:
    cparts, separts = [], []
    for m in models:
        if m is not None and var in m.params:
            cparts.append(fmt_c(m.params[var], m.pvalues[var]))
            separts.append(fmt_se(m.bse[var]))
        else:
            cparts.append('')
            separts.append('')
    rows.append(label + ' & ' + ' & '.join(cparts) + r' \\')
    rows.append(r'& ' + ' & '.join(separts) + r' \\')
    rows.append(r'\addlinespace[2pt]')
rows += [
    r'\midrule',
    f'Observations & {int(m_v1.nobs):,} & {int(m_v2.nobs):,} & {int(m_v3.nobs):,} \\\\',
    f'$R^2$ & {m_v1.rsquared:.3f} & {m_v2.rsquared:.3f} & {m_v3.rsquared:.3f} \\\\',
    r'Upazila FEs & Yes & Yes & Yes \\',
    r'Season FEs  & Yes & Yes & Yes \\',
    r'\addlinespace[2pt]',
    r'\multicolumn{4}{p{0.75\textwidth}}{\footnotesize \textit{Notes:}'
    r' Unit of observation is trader $\times$ season, restricted to'
    r' trader-seasons with $\geq 2$ active procurement months ($N='
    + str(len(pv)) + r'$).'
    r' Price CV $=$ (SD/mean) $\times$ 100.'
    r' SE clustered at upazila level.'
    r' $^{*}p<0.10$, $^{**}p<0.05$, $^{***}p<0.01$.} \\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(TABLES / 'tab10_price_variance.tex', rows)

In [ ]:
# ── tab_app_upazila.tex  (Appendix A) ────────────────────────────────────────
# Self-contained longtable with caption+label; main.tex just does \input{...}
header = [
    r'Upazila & $N$ & Any credit & Credit & On & Adv.\ to & Adv.\ to \\',
    r'& & (\%) & days & delivery & traders & farmers \\',
]
rows = [
    r'\begin{longtable}{lrrrrrr}',
    r'\caption{Upazila-Level Credit Prevalence (Paddy)} \label{tab:upazila} \\',
    r'\toprule', *header, r'\midrule',
    r'\endfirsthead',
    r'\multicolumn{7}{l}{\tablename\ \thetable{} -- continued} \\',
    r'\toprule', *header, r'\midrule',
    r'\endhead',
    r'\midrule \multicolumn{7}{r}{\footnotesize Continued on next page} \\',
    r'\endfoot',
    r'\bottomrule',
    r'\endlastfoot',
]

def fmt_val(v):
    return '{---}' if pd.isna(v) else f'{v:.1f}'

# Confidentiality: publish only upazilas with at least MIN_TRADERS distinct traders
upz_pub = upazila_credit[upazila_credit['n_traders'] >= MIN_TRADERS]

for upazila, row in upz_pub.iterrows():
    rows.append(
        f'{upazila} & {int(row["N"])} & {fmt_val(row["pct_any_credit"])}'
        f' & {fmt_val(row["mean_credit_days"])} & {fmt_val(row["mean_pct_delivery"])}'
        f' & {fmt_val(row["mean_pct_adv_traders"])} & {fmt_val(row["mean_pct_adv_farmers"])} \\\\'
    )
rows.append(r'\end{longtable}')
write_tex(TABLES / 'tab_app_upazila.tex', rows)
print(f'tab_app_upazila done  ({len(upz_pub)} of {len(upazila_credit)} upazilas; '
      f'{len(upazila_credit) - len(upz_pub)} with < {MIN_TRADERS} traders omitted).')

In [ ]:
# ── tab_app_firststage.tex  (Appendix B) ─────────────────────────────────────
# First-stage regression: any_credit ~ LOO instrument + district + season + month FEs
c_fs  = fs.params['loo_credit_rate']
se_fs = fs.bse['loo_credit_rate']
p_fs  = fs.pvalues['loo_credit_rate']

rows = [
    r'\begin{tabular}{lc}',
    r'\toprule',
    r'& Any supplier credit \\',
    r'\midrule',
    r'LOO upazila-season credit rate & ' + fmt_c(c_fs, p_fs) + r' \\',
    r'& ' + fmt_se(se_fs) + r' \\',
    r'\addlinespace[4pt]',
    r'\midrule',
    f'Observations & {int(fs.nobs):,} \\\\',
    f'$R^2$ & {fs.rsquared:.3f} \\\\',
    r'District FEs & Yes \\',
    r'Season \& month FEs & Yes \\',
    f'$F$-statistic (instrument) & {fstat_instrument:.1f} \\\\',
    r'\addlinespace[2pt]',
    r'\multicolumn{2}{p{0.55\textwidth}}{\footnotesize \textit{Notes:} '
    r'First stage of the 2SLS in Table~\ref{tab:iv}. '
    r'Dependent variable: binary indicator for any supplier credit. '
    r'SE clustered at upazila level. '
    r'$^{*}p<0.10$, $^{**}p<0.05$, $^{***}p<0.01$.} \\',
    r'\bottomrule',
    r'\end{tabular}',
]
write_tex(f'{TABLES}/tab_app_firststage.tex', rows)
print('tab_app_firststage done.')

In [ ]:
# ── fig1_credit_prevalence.pdf  (2×2: panels A–D) ────────────────────────────
# main.tex describes panels A-D in a single figure; we export all four here.
fig1, axes = plt.subplots(2, 2, figsize=(12, 9))

type_order = ['Faria/bepari', 'Aratdar', 'Broker/intermediary', 'Other']
s_order    = ['Aman 2016/17', 'Boro 2017', 'Aus 2017']

# Panel A: credit prevalence by trader type
pct_by_type = (
    paddy_cr.dropna(subset=['any_credit_paddy', 'trader_type'])
    .groupby('trader_type')['any_credit_paddy']
    .agg(['mean', 'count'])
    .reset_index()
)
pct_by_type['mean'] *= 100
pct_by_type = pct_by_type.set_index('trader_type').reindex(type_order).dropna().reset_index()

axes[0, 0].barh(pct_by_type['trader_type'], pct_by_type['mean'], color='steelblue', height=0.5)
axes[0, 0].set_xlabel('Trader-seasons with any supplier credit (%)')
axes[0, 0].set_title('A. Credit prevalence by trader type')
for i, (v, n) in enumerate(zip(pct_by_type['mean'], pct_by_type['count'])):
    axes[0, 0].text(v + 0.5, i, f'{v:.1f}%  (N={int(n)})', va='center', fontsize=8)
axes[0, 0].set_xlim(0, 110)
axes[0, 0].invert_yaxis()

# Panel B: credit prevalence by season
pct_by_season = (
    paddy_cr.dropna(subset=['any_credit_paddy', 'season'])
    .groupby('season')['any_credit_paddy']
    .agg(['mean', 'count'])
    .reset_index()
)
pct_by_season['mean'] *= 100
pct_by_season = pct_by_season.set_index('season').reindex(s_order).dropna().reset_index()

axes[0, 1].barh(pct_by_season['season'], pct_by_season['mean'], color='coral', height=0.4)
axes[0, 1].set_xlabel('Trader-seasons with any supplier credit (%)')
axes[0, 1].set_title('B. Credit prevalence by season')
for i, (v, n) in enumerate(zip(pct_by_season['mean'], pct_by_season['count'])):
    axes[0, 1].text(v + 0.5, i, f'{v:.1f}%  (N={int(n)})', va='center', fontsize=8)
axes[0, 1].set_xlim(0, 110)
axes[0, 1].invert_yaxis()

# Panel C: payment mix by trader type
pay_cols = {'d39': 'On delivery', 'd40': 'Adv. to traders', 'd41': 'Adv. to farmers'}
pay_by_type = (
    paddy_cr.dropna(subset=['trader_type'])
    .groupby('trader_type')[list(pay_cols.keys())]
    .mean()
    .rename(columns=pay_cols)
    .reindex(type_order)
    .dropna(how='all')
)
x     = range(len(pay_by_type))
width = 0.25
colors = ['#2196F3', '#FF9800', '#4CAF50']
for i, (col, color) in enumerate(zip(pay_by_type.columns, colors)):
    axes[1, 0].bar([xi + i * width for xi in x], pay_by_type[col], width, label=col, color=color, alpha=0.85)
axes[1, 0].set_xticks([xi + width for xi in x])
axes[1, 0].set_xticklabels(pay_by_type.index, rotation=15, ha='right', fontsize=8)
axes[1, 0].set_ylabel('Mean % of purchases')
axes[1, 0].set_title('C. Payment mix by trader type')
axes[1, 0].legend(fontsize=8)

# Panel D: payment mix by season
pay_by_season = (
    paddy_cr.dropna(subset=['season'])
    .groupby('season')[list(pay_cols.keys())]
    .mean()
    .rename(columns=pay_cols)
    .reindex(s_order)
    .dropna(how='all')
)
x2 = range(len(pay_by_season))
for i, (col, color) in enumerate(zip(pay_by_season.columns, colors)):
    axes[1, 1].bar([xi + i * width for xi in x2], pay_by_season[col], width, label=col, color=color, alpha=0.85)
axes[1, 1].set_xticks([xi + width for xi in x2])
axes[1, 1].set_xticklabels(pay_by_season.index, rotation=15, ha='right', fontsize=8)
axes[1, 1].set_ylabel('Mean % of purchases')
axes[1, 1].set_title('D. Payment mix by season')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
fig1.savefig(f'{FIGURES}/fig1_credit_prevalence.pdf', bbox_inches='tight')
plt.close(fig1)
print('fig1_credit_prevalence.pdf saved (2×2 panels A–D).')

In [ ]:
# ── fig2_payment_mix.pdf  (panels C + D) ────────────────────────────────────
pay_cols = {'d39': 'On delivery', 'd40': 'Adv. to traders', 'd41': 'Adv. to farmers'}

fig2, axes = plt.subplots(1, 2, figsize=(10, 4))

# Panel C: payment mix by trader type
pay_by_type = (
    paddy_cr.dropna(subset=['trader_type'])
    .groupby('trader_type')[list(pay_cols.keys())]
    .mean()
    .rename(columns=pay_cols)
    .reindex(type_order)
    .dropna(how='all')
)
x     = range(len(pay_by_type))
width = 0.25
colors = ['#2196F3', '#FF9800', '#4CAF50']
for i, (col, color) in enumerate(zip(pay_by_type.columns, colors)):
    axes[0].bar([xi + i * width for xi in x], pay_by_type[col], width, label=col, color=color, alpha=0.85)
axes[0].set_xticks([xi + width for xi in x])
axes[0].set_xticklabels(pay_by_type.index, rotation=15, ha='right', fontsize=8)
axes[0].set_ylabel('Mean % of purchases')
axes[0].set_title('C. Payment mix by trader type')
axes[0].legend(fontsize=8)

# Panel D: payment mix by season
pay_by_season = (
    paddy_cr.dropna(subset=['season'])
    .groupby('season')[list(pay_cols.keys())]
    .mean()
    .rename(columns=pay_cols)
    .reindex(s_order)
    .dropna(how='all')
)
x2 = range(len(pay_by_season))
for i, (col, color) in enumerate(zip(pay_by_season.columns, colors)):
    axes[1].bar([xi + i * width for xi in x2], pay_by_season[col], width, label=col, color=color, alpha=0.85)
axes[1].set_xticks([xi + width for xi in x2])
axes[1].set_xticklabels(pay_by_season.index, rotation=15, ha='right', fontsize=8)
axes[1].set_ylabel('Mean % of purchases')
axes[1].set_title('D. Payment mix by season')
axes[1].legend(fontsize=8)

plt.tight_layout()
fig2.savefig(f'{FIGURES}/fig2_payment_mix.pdf', bbox_inches='tight')
plt.close(fig2)
print('fig2_payment_mix.pdf saved.')

In [ ]:
# ── fig3_implicit_rate.pdf ───────────────────────────────────────────────────
fig3, axes = plt.subplots(1, 2, figsize=(10, 4))

rate_clipped = impl['implied_annual_rate_pct'].clip(upper=200)
median_rate  = impl['implied_annual_rate_pct'].median()
mean_rate    = impl['implied_annual_rate_pct'].mean()

# Panel A: histogram
axes[0].hist(rate_clipped, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(median_rate, color='red',    linestyle='--', linewidth=1.5, label=f'Median = {median_rate:.0f}%')
axes[0].axvline(mean_rate,   color='orange', linestyle=':',  linewidth=1.5, label=f'Mean = {mean_rate:.0f}%')
axes[0].set_xlabel('Implied annual interest rate (%)\n(capped at 200%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('A. Distribution of implicit interest rates')
axes[0].legend(fontsize=8)

# Panel B: rate vs credit duration scatter
axes[1].scatter(impl['credit_days'], rate_clipped,
                alpha=0.6, color='coral', s=40, edgecolors='white', linewidths=0.4)
axes[1].set_xlabel('Credit duration (days)')
axes[1].set_ylabel('Implied annual rate (%)\n(capped at 200%)')
axes[1].set_title('B. Implicit rate vs. credit duration')

# Annotations
axes[1].text(0.97, 0.97,
             f'N = {len(impl)}\nMedian = {median_rate:.0f}%',
             transform=axes[1].transAxes, ha='right', va='top', fontsize=8,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
fig3.savefig(f'{FIGURES}/fig3_implicit_rate.pdf', bbox_inches='tight')
plt.close(fig3)
print('fig3_implicit_rate.pdf saved.')
print()
print('All paper output complete.')
print(f'  Tables : {TABLES}/')
print(f'  Figures: {FIGURES}/')